In [1]:
import numpy as np
import pandas as pd

import ast

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from nltk.stem.porter import PorterStemmer

import pickle

In [2]:
pip install nltk

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [4]:
movies = pd.read_csv("tmdb_5000_movies.csv")
credits = pd.read_csv("tmdb_5000_credits.csv")

In [5]:
movies.head()

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124


In [6]:
movies.head()

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124


In [7]:
movies = movies.merge(credits,on="title")

In [8]:
movies = movies[
[
'title',
'overview',
'genres',
'keywords',
'cast',
'crew'
]
]

In [9]:
movies.isnull().sum()

title       0
overview    3
genres      0
keywords    0
cast        0
crew        0
dtype: int64

In [10]:
movies.dropna(inplace=True)

In [11]:
def convert(text):

    L=[]

    for i in ast.literal_eval(text):
        L.append(i['name'])

    return L

In [12]:
movies['genres']=movies['genres'].apply(convert)

In [13]:
movies['keywords']=movies['keywords'].apply(convert)

In [14]:
def convert_cast(text):

    L=[]

    counter=0

    for i in ast.literal_eval(text):

        if counter!=3:
            L.append(i['name'])
            counter+=1
        else:
            break

    return L

In [15]:
movies['cast']=movies['cast'].apply(convert_cast)

In [16]:
def fetch_director(text):

    L=[]

    for i in ast.literal_eval(text):

        if i['job']=="Director":
            L.append(i['name'])

    return L

In [17]:
movies['crew']=movies['crew'].apply(fetch_director)

In [18]:
movies['overview']=movies['overview'].apply(lambda x:x.split())

In [19]:
movies['genres']=movies['genres'].apply(lambda x:[i.replace(" ","") for i in x])

movies['keywords']=movies['keywords'].apply(lambda x:[i.replace(" ","") for i in x])

movies['cast']=movies['cast'].apply(lambda x:[i.replace(" ","") for i in x])

movies['crew']=movies['crew'].apply(lambda x:[i.replace(" ","") for i in x])

In [20]:
movies['tags']=movies['overview']+movies['genres']+movies['keywords']+movies['cast']+movies['crew']

In [21]:
new_df=movies[['title','tags']]

In [23]:
new_df.loc[:, 'tags'] = new_df['tags'].apply(lambda x: " ".join(x))

In [24]:
new_df.loc[:, 'tags']=new_df['tags'].apply(lambda x:x.lower())

In [25]:
ps=PorterStemmer()

In [26]:
def stem(text):

    y=[]

    for i in text.split():

        y.append(ps.stem(i))

    return " ".join(y)

In [28]:
new_df.loc[:, 'tags']=new_df['tags'].apply(stem)

In [29]:
cv=CountVectorizer(max_features=5000,stop_words='english')

In [32]:
new_df.head()

,title,tags
0,Avatar,"i n t h e 2 2 n d c e n t u r y , a p a r a p ..."
1,Pirates of the Caribbean: At World's End,"c a p t a i n b a r b o s s a , l o n g b e l ..."
2,Spectre,a c r y p t i c m e s s a g e f r o m b o n d ...
3,The Dark Knight Rises,f o l l o w i n g t h e d e a t h o f d i s t ...
4,John Carter,"j o h n c a r t e r i s a w a r - w e a r y , ..."


In [33]:
new_df['tags'].isnull().sum()
(new_df['tags']=="").sum()

np.int64(0)

In [34]:
new_df['tags'].apply(type).value_counts()

tags
<class 'str'>    4806
Name: count, dtype: int64

In [38]:
new_df.loc[:, 'tags'] = new_df['tags'].apply(lambda x:" ".join(x))

In [39]:
new_df.head()

,title,tags
0,Avatar,i n t h e 2 ...
1,Pirates of the Caribbean: At World's End,c a p t a i ...
2,Spectre,a c r y p t ...
3,The Dark Knight Rises,f o l l o w ...
4,John Carter,j o h n c a ...


In [40]:
new_df = new_df[new_df['tags'].str.strip()!=""]

In [43]:
new_df.head()

,title,tags
0,Avatar,i n t h e 2 ...
1,Pirates of the Caribbean: At World's End,c a p t a i ...
2,Spectre,a c r y p t ...
3,The Dark Knight Rises,f o l l o w ...
4,John Carter,j o h n c a ...


In [44]:
print(new_df['tags'].iloc[0])

i       n       t       h       e       2       2       n       d       c       e       n       t       u       r       y       ,       a       p       a       r       a       p       l       e       g       i       c       m       a       r       i       n       e       i       s       d       i       s       p       a       t       c       h       e       d       t       o       t       h       e       m       o       o       n       p       a       n       d       o       r       a       o       n       a       u       n       i       q       u       e       m       i       s       s       i       o       n       ,       b       u       t       b       e       c       o       m       e       s       t       o       r       n       b       e       t       w       e       e       n       f       o       l       l       o       w       i       n       g       o       r       d       e       r       s       a       n       d       p       r       o       t       e       c       t       

In [45]:
new_df['tags'].apply(type).value_counts()

tags
<class 'str'>    4806
Name: count, dtype: int64

In [46]:
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))

In [48]:
new_df['tags'].apply(len).describe()

count     4806.000000
mean     12532.921764
std       4804.728417
min        609.000000
25%       8897.000000
50%      12129.000000
75%      15289.000000
max      42849.000000
Name: tags, dtype: float64

In [49]:
new_df = movies[['title','tags']].copy()

new_df['tags'] = new_df['tags'].apply(
    lambda x: " ".join(x)
)

new_df['tags'] = new_df['tags'].apply(
    lambda x: x.lower()
)

In [50]:
new_df.head()

,title,tags
0,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,Spectre,a cryptic message from bond’s past sends him o...
3,The Dark Knight Rises,following the death of district attorney harve...
4,John Carter,"john carter is a war-weary, former military ca..."


In [51]:
cv = CountVectorizer(
    max_features=5000,
    stop_words='english'
)

vectors = cv.fit_transform(new_df['tags']).toarray()

In [52]:
similarity=cosine_similarity(vectors)

In [53]:
def recommend(movie):

    movie_index = new_df[new_df['title'] == movie].index[0]

    distances = similarity[movie_index]

    movie_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x:x[1]
    )[1:6]

    for i in movie_list:

        print(new_df.iloc[i[0]].title)

In [54]:
recommend("Avatar")

Titan A.E.
Small Soldiers
Independence Day
Ender's Game
Aliens vs Predator: Requiem


In [55]:
pickle.dump(new_df,open("movies.pkl","wb"))

In [56]:
pickle.dump(similarity,open("similarity.pkl","wb"))

In [65]:
import requests

def fetch_poster(movie_id):

    url=f"https://api.themoviedb.org/3/movie/{movie_id}?api_key=YOUR_API_KEY&language=en-US"

    data=requests.get(url).json()

    poster_path=data['poster_path']

    return "https://image.tmdb.org/t/p/w500/"+poster_path